# P01 (basic) — Homogene Transformationen & Ray-Casting-Selektion

**Modul 19 — 3D User Interfaces**

Jede 3D-Interaktion steht und faellt mit zwei Bausteinen:
1. **Koordinaten-Transformationen** — ein Objekt existiert gleichzeitig in mehreren Bezugssystemen
   (Objekt / Welt / Kamera), und Interaktion heisst staendig, zwischen ihnen umzurechnen.
2. **Ray-Casting** — die haeufigste Selektionstechnik: ein virtueller Strahl vom Controller,
   selektiert wird das **erste getroffene Objekt**.

In diesem Projekt baust du beide von Grund auf: die **4x4-Transformationsmatrizen** (und siehst,
warum man homogene Koordinaten braucht) und die **Ray-Kugel-Schnittmathematik**, mit der du eine
Szene per Strahl selektierst.

### Ziel
Nach diesem Projekt kannst du …
- homogene 4x4-Matrizen fuer Translation/Rotation/Skalierung bauen und **verketten**,
- Punkte durch die Transformationskette Objekt->Welt schicken und die Kette **invertieren**,
- die **Ray-Kugel-Schnittgleichung** herleiten und implementieren,
- eine 3D-Szene per Ray-Casting selektieren (naechster Treffer) und das visualisieren.

### Format
Jupyter-Notebook — die 3D-Geometrie lebt von Zahlen + Visualisierung nebeneinander.

### Vorwissen
Lineare Algebra (Matrix x Vektor, Skalarprodukt). Kapitel 3 & 5 des Modul-19-Skripts.
Die Rotationsmathematik aus Modul 17 (Quaternionen -> Rotationsmatrix) taucht hier wieder auf.

### Aufgaben
Die meisten Zellen sind vorgegeben; an den `# TODO`-Stellen fuellst du die Kernbausteine ein.
Vollstaendige Loesung in `solution/`.


## Setup
Wir nutzen `numpy` (Mathematik) und `matplotlib` (3D-Plot). Beides ist in der `.venv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (aktiviert 3D-Projektion)
np.set_printoptions(precision=3, suppress=True)


## Teil A — Homogene 4x4-Transformationen

Ein 3D-Punkt $\mathbf{p}=(x,y,z)$ wird **homogen** als $(x,y,z,1)$ geschrieben. Der Grund
(Skript, Kap. 3): **Translation ist keine lineare Abbildung** und laesst sich nur so als Matrix
schreiben — dann sind Rotation, Translation, Skalierung **alle** 4x4-Matrizen und **verkettbar**.

$$T(\mathbf t)=\begin{pmatrix}1&0&0&t_x\\0&1&0&t_y\\0&0&1&t_z\\0&0&0&1\end{pmatrix},\quad
S(\mathbf s)=\mathrm{diag}(s_x,s_y,s_z,1),\quad
R=\begin{pmatrix}\mathbf R_{3\times3}&\mathbf 0\\\mathbf 0^\top&1\end{pmatrix}$$

**Deine Aufgabe:** Fuelle die Translations- und Skalierungsmatrix aus. `rotation_matrix` (aus einer
Achse + Winkel, via Rodrigues) ist vorgegeben.

In [ ]:
def translation(t):
    # TODO: 4x4-Translationsmatrix fuer den Vektor t=(tx,ty,tz)
    M = np.eye(4)
    ...  # TODO: setze die Translationsspalte
    return M

def scaling(s):
    # TODO: 4x4-Skalierungsmatrix fuer s=(sx,sy,sz)
    M = np.eye(4)
    ...  # TODO
    return M

def rotation_matrix(axis, angle_deg):
    # Rodrigues-Formel: 3x3-Rotation um eine (beliebige) Achse, eingebettet in 4x4. [vorgegeben]
    axis = np.asarray(axis, float); axis = axis / np.linalg.norm(axis)
    th = np.deg2rad(angle_deg)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    R3 = np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * (K @ K)
    M = np.eye(4); M[:3, :3] = R3
    return M

def to_homogeneous(pts):
    # (N,3) -> (N,4) mit w=1   [vorgegeben]
    pts = np.atleast_2d(pts)
    return np.hstack([pts, np.ones((pts.shape[0], 1))])

def from_homogeneous(pts_h):
    # (N,4) -> (N,3), teilt durch w   [vorgegeben]
    return pts_h[:, :3] / pts_h[:, 3:4]

# Test: verkette erst skalieren, dann rotieren (90 deg um z), dann verschieben
M = translation([1, 2, 3]) @ rotation_matrix([0, 0, 1], 90) @ scaling([2, 2, 2])
p = np.array([[1, 0, 0]])
p_out = from_homogeneous((M @ to_homogeneous(p).T).T)
print("transformierter Punkt:", p_out[0])


**Erwartung / Selbstcheck.** Der Punkt $(1,0,0)$ wird: skaliert ->$(2,0,0)$, um 90 Grad um z
gedreht ->$(0,2,0)$, verschoben um $(1,2,3)$ -> **$(1,4,3)$**. Kommt das heraus?

Beachte: `translation @ rotation @ scaling` wendet **von rechts nach links** an — erst Skalierung.
Vertausche die Reihenfolge und du bekommst etwas anderes (Matrixmultiplikation ist **nicht
kommutativ** — wie die Rotationen in Modul 17).

## Teil B — Transformationskette und ihre Inverse

Ein Objekt (lokale Koordinaten) sitzt ueber eine **Model-Matrix** $M$ in der Welt. Umgekehrt
rechnet man Weltkoordinaten mit $M^{-1}$ zurueck ins Objekt. Fuer eine starre Transformation
(Rotation + Translation) gilt die geschlossene Form (Skript, Kap. 3):

$$M=\begin{pmatrix}\mathbf R&\mathbf t\\\mathbf 0^\top&1\end{pmatrix},\qquad
  M^{-1}=\begin{pmatrix}\mathbf R^\top&-\mathbf R^\top\mathbf t\\\mathbf 0^\top&1\end{pmatrix}$$

(nutzt $\mathbf R^{-1}=\mathbf R^\top$, weil Rotationsmatrizen orthogonal sind — kein numerisches
Invertieren noetig).

**Deine Aufgabe:** Implementiere `rigid_inverse(M)` mit dieser Formel (nicht `np.linalg.inv`!).

In [ ]:
def rigid_inverse(M):
    R = M[:3, :3]
    t = M[:3, 3]
    Minv = np.eye(4)
    # TODO: fuelle Rotationsblock (R^T) und Translationsteil (-R^T t)
    ...  # TODO
    return Minv

# Verifikation gegen np.linalg.inv (nur zur Kontrolle)
M = translation([1, 2, 3]) @ rotation_matrix([1, 1, 0], 37)
Minv = rigid_inverse(M)
print("max Abweichung von np.linalg.inv:", np.abs(Minv - np.linalg.inv(M)).max())
print("M @ Minv == I ?", np.allclose(M @ Minv, np.eye(4)))


**Erwartung.** Die Abweichung von `np.linalg.inv` ist ~$10^{-15}$ (Maschinengenauigkeit) und
$M M^{-1}=I$. Die geschlossene Formel ist schneller und numerisch stabiler als allgemeines
Invertieren — 3D-Engines nutzen sie ueberall.

## Teil C — Ray-Kugel-Schnitt (die Selektionsmathematik)

Ein **Strahl** ist $\mathbf r(t)=\mathbf o + t\,\mathbf d$ mit Ursprung $\mathbf o$ und
**normierter** Richtung $\mathbf d$ (dann ist $t$ = Distanz). Objekte modellieren wir als
**Bounding Spheres** (Zentrum $\mathbf c$, Radius $R$). Einsetzen in $\|\mathbf p-\mathbf c\|^2=R^2$
ergibt (Skript, Kap. 5) mit $\mathbf m=\mathbf o-\mathbf c$:

$$t^2 + 2t\,(\mathbf m\cdot\mathbf d) + (\|\mathbf m\|^2-R^2)=0,\qquad
  b=\mathbf m\cdot\mathbf d,\ c=\|\mathbf m\|^2-R^2,\ \Delta=b^2-c.$$

Kein Treffer, wenn $\Delta<0$. Sonst ist der naehere (vordere) Schnitt $t=-b-\sqrt\Delta$.

**Deine Aufgabe:** Implementiere `ray_sphere(o, d, c, R)` -> Distanz $t>0$ oder `None`.

In [ ]:
def ray_sphere(o, d, c, R):
    o = np.asarray(o, float); d = np.asarray(d, float); c = np.asarray(c, float)
    d = d / np.linalg.norm(d)          # sicherstellen: normiert
    m = o - c
    b = m @ d
    cc = m @ m - R * R
    disc = b * b - cc                   # Diskriminante
    # TODO: kein Treffer -> None; sonst naeheren positiven t zurueckgeben
    ...  # TODO
    return None

# Mini-Test: Strahl entlang +x trifft Kugel bei (5,0,0), R=1 -> erste Beruehrung bei t=4
print("t =", ray_sphere([0, 0, 0], [1, 0, 0], [5, 0, 0], 1.0))   # erwartet 4.0
print("miss:", ray_sphere([0, 0, 0], [0, 1, 0], [5, 0, 0], 1.0)) # erwartet None


**Erwartung.** Der Strahl entlang $+x$ vom Ursprung trifft die Kugel um $(5,0,0)$ mit Radius 1
zuerst bei **$t=4$**; ein Strahl entlang $+y$ verfehlt sie (`None`).

## Teil D — Eine Szene per Ray-Casting selektieren

Jetzt alles zusammen: eine Szene aus mehreren Kugel-Objekten, ein Zeigestrahl, und wir waehlen
das **naechste getroffene** Objekt (kleinstes $t>0$). Die Szene und der Plot sind vorgegeben; du
implementierst nur die Selektionsschleife `select(o, d, scene)`.

In [ ]:
# Szene: Liste von (name, center, radius)   [vorgegeben]
scene = [
    ("A", np.array([3.0,  0.5, 0.0]), 0.6),
    ("B", np.array([6.0, -0.3, 0.4]), 0.8),
    ("C", np.array([4.5,  2.0, 0.0]), 0.5),
    ("D", np.array([8.0,  0.0, -1.0]), 0.7),
]
ray_o = np.array([0.0, 0.0, 0.0])
ray_d = np.array([1.0, 0.02, 0.05]); ray_d = ray_d / np.linalg.norm(ray_d)

def select(o, d, scene):
    # TODO: gib (name, t) des naechsten getroffenen Objekts zurueck, oder (None, None)
    best_name, best_t = None, np.inf
    ...  # TODO: iteriere ueber scene, nutze ray_sphere, behalte kleinstes t
    return (best_name, None if best_name is None else best_t)

name, t = select(ray_o, ray_d, scene)
print(f"selektiert: {name} bei t={t}")


**Erwartung.** Der Strahl zeigt fast entlang $+x$; er trifft **A** (bei ~$t\approx3$) zuerst,
obwohl B und D weiter hinten auch getroffen wuerden — Ray-Casting selektiert das **naechste**
Objekt. C liegt zu weit oben und wird verfehlt. Der Plot zeigt Strahl (rot), getroffenes Objekt
(gruen), Rest (grau).

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
u, v = np.mgrid[0:2*np.pi:16j, 0:np.pi:8j]
for nm, c, R in scene:
    xs = c[0] + R*np.cos(u)*np.sin(v); ys = c[1] + R*np.sin(u)*np.sin(v); zs = c[2] + R*np.cos(v)
    hit = (nm == name)
    ax.plot_surface(xs, ys, zs, color="green" if hit else "lightgray",
                    alpha=0.9 if hit else 0.35)
    ax.text(c[0], c[1], c[2]+R+0.3, nm, fontsize=11)
L = t if t is not None else 9.0
line = ray_o[:, None] + ray_d[:, None]*np.linspace(0, L+1, 2)
ax.plot(line[0], line[1], line[2], "r-", lw=2, label="Zeigestrahl")
if t is not None:
    hit_pt = ray_o + ray_d*t
    ax.scatter(*hit_pt, color="red", s=60)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title(f"Ray-Casting: selektiert {name}"); ax.legend()
try:
    ax.set_box_aspect((1, 1, 1))
except Exception:
    pass
plt.show()


## Fazit

Du hast die zwei geometrischen Fundamente jedes 3DUI gebaut:
- **Homogene Transformationen** (verkettbar, invertierbar in geschlossener Form),
- **Ray-Casting-Selektion** (Ray-Kugel-Schnitt + naechster Treffer).

Das war die *ideale* Welt: kein Rauschen, Objekte perfekt getroffen. Im **Medium-Projekt** kommt
die Realitaet dazu — **Hand-Tremor** macht die Zeigerichtung verrauscht, und du wirst sehen, dass
**ferne, kleine Ziele** dramatisch schwerer zu treffen sind (angulares Fitts' Law) und wie
**Go-Go** die Reichweite rettet. Das **Final-Projekt** vergleicht dann ganze Selektionstechniken
unter realistischem Gedraenge.